# CommGuard calibration v4 sampling study

This diagnostic notebook studies the PCIe telemetry resolution boundary on Kaggle `GPU T4 x2`.

It does **not** replace the canonical calibration-v3 gate and must not be used directly as input to the benign-corpus notebook.

Study design:

- exact immutable CommGuard source checkout;
- 5 idle repetitions;
- payloads: 16, 32, 48, 64, 96, and 128 MiB;
- 5 repetitions per payload;
- deterministic payload-order shuffling within each repetition;
- 30-second measured interval for idle and collective runs;
- at least 30 expected NVML polling opportunities per run;
- unchanged calibration thresholds: 80% capture rate, 3× idle baseline, and 1 MB/s floor.

Expected total: **35 two-rank runs**. Preserve partial or failed evidence rather than rerunning cells in the same workspace.


In [ ]:
import importlib
import os
from pathlib import Path
import re
import subprocess
import sys

NOTEBOOK_VERSION = "commguard_calibration_v4_sampling_study"
REPOSITORY_URL = "https://github.com/waqasm86/CommGuard.git"
REVIEWED_COMMIT = "5af57bc220cddd750a1931a3bd52c4da24f990a0"
REPOSITORY = Path("/kaggle/working/commguard-source")

if not re.fullmatch(r"[0-9a-f]{40}", REVIEWED_COMMIT):
    raise RuntimeError("REVIEWED_COMMIT must be an immutable 40-character SHA.")

if not REPOSITORY.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)],
        check=True,
    )

if not (REPOSITORY / ".git").is_dir():
    raise RuntimeError(f"Refusing non-Git source directory: {REPOSITORY}")

subprocess.run(
    ["git", "-C", str(REPOSITORY), "fetch", "origin", REVIEWED_COMMIT],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPOSITORY), "checkout", "--detach", REVIEWED_COMMIT],
    check=True,
)

head = subprocess.run(
    ["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

dirty = subprocess.run(
    ["git", "-C", str(REPOSITORY), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

pushed_refs = subprocess.run(
    ["git", "-C", str(REPOSITORY), "branch", "-r", "--contains", head],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if head != REVIEWED_COMMIT or dirty or not pushed_refs:
    raise RuntimeError(
        "Reproducibility gate failed: "
        f"head={head} dirty={bool(dirty)} pushed={bool(pushed_refs)}"
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-build-isolation",
        "--no-deps",
        "-e",
        str(REPOSITORY),
    ],
    check=True,
)

SOURCE_ROOT = (REPOSITORY / "src").resolve()
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + (
    os.pathsep + existing_pythonpath if existing_pythonpath else ""
)

sys.path[:] = [
    entry
    for entry in sys.path
    if Path(entry or ".").resolve() != SOURCE_ROOT
]
sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()

for module_name in [
    name
    for name in list(sys.modules)
    if name == "commguard" or name.startswith("commguard.")
]:
    del sys.modules[module_name]

import commguard

commguard_path = Path(commguard.__file__).resolve()
try:
    commguard_path.relative_to(SOURCE_ROOT)
except ValueError as exc:
    raise RuntimeError(
        f"CommGuard imported outside reviewed source: {commguard_path}"
    ) from exc

print(
    {
        "reviewed_commit": head,
        "remote_refs": pushed_refs.splitlines(),
        "commguard_import": str(commguard_path.relative_to(REPOSITORY)),
        "torchrun_pythonpath_prefix": os.environ["PYTHONPATH"].split(os.pathsep)[0],
    }
)


In [ ]:
import shutil
import torch

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("nvidia-smi is required.")

gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,compute_cap",
        "--format=csv,noheader,nounits",
    ],
    check=True,
    capture_output=True,
    text=True,
)

GPU_SUMMARY = []
for line in gpu_query.stdout.splitlines():
    index, name, memory_mib, compute_capability = [
        part.strip() for part in line.split(",")
    ]
    GPU_SUMMARY.append(
        {
            "index": int(index),
            "name": name,
            "memory_total_mib": int(memory_mib),
            "compute_capability": compute_capability,
        }
    )

if len(GPU_SUMMARY) != 2 or any(
    "T4" not in gpu["name"] for gpu in GPU_SUMMARY
):
    raise RuntimeError(
        f"Expected exactly two Tesla T4 GPUs, observed: {GPU_SUMMARY}"
    )

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("PyTorch must expose exactly two CUDA devices.")

if not torch.distributed.is_nccl_available():
    raise RuntimeError("The PyTorch build does not expose NCCL.")

print(
    {
        "gpus": GPU_SUMMARY,
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "nccl_available": torch.distributed.is_nccl_available(),
    }
)


In [ ]:
from datetime import datetime, timezone

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ARTIFACTS = Path(
    f"/kaggle/working/commguard-calibration-v4-{NOTEBOOK_RUN_ID}"
)

if ARTIFACTS.exists():
    raise RuntimeError(
        f"Refusing reused workspace: {ARTIFACTS}. Start a fresh notebook run."
    )

ARTIFACTS.mkdir(parents=True, exist_ok=False)
probe = ARTIFACTS / ".write-probe"
probe.write_text("writable\n", encoding="utf-8")
probe.unlink()

print(
    {
        "notebook_run_id": NOTEBOOK_RUN_ID,
        "artifact_workspace": str(ARTIFACTS),
    }
)


In [ ]:
from commguard.artifacts import ArtifactStore
from commguard.environment.preflight import (
    check_environment,
    summarize_environment,
)
from commguard.provenance import ProvenanceContext
from commguard.schemas import CURRENT_SCHEMA_VERSION, SCHEMA_VERSION

STORE = ArtifactStore(ARTIFACTS)
STORE.initialize()

CONTEXT = ProvenanceContext.create(
    corpus_id=f"corpus-calibration-v4-{NOTEBOOK_RUN_ID}",
    experiment_session_id=f"session-calibration-v4-{NOTEBOOK_RUN_ID}",
    collection_id=f"collection-calibration-v4-{NOTEBOOK_RUN_ID}",
    notebook_version=NOTEBOOK_VERSION,
    input_archive_sha256=None,
    random_seed=20260803,
    repository_root=REPOSITORY,
)

if CONTEXT.source_dirty or CONTEXT.source_commit != REVIEWED_COMMIT:
    raise RuntimeError(
        "SDK provenance no longer matches the clean reviewed source commit."
    )

BOOTSTRAP = {
    "artifact_kind": "experiment_summary",
    "schema_version": SCHEMA_VERSION,
    "summary_type": "calibration_v4_notebook_bootstrap",
    "notebook_version": NOTEBOOK_VERSION,
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "reviewed_commit": REVIEWED_COMMIT,
    "source_repository": "commguard-source",
    "gpu_summary": GPU_SUMMARY,
    "artifact_schema_version": CURRENT_SCHEMA_VERSION,
    "study_role": "diagnostic_sampling_resolution_study",
    "not_a_standard_calibration_gate": True,
}
STORE.write_json("environment/notebook-bootstrap.json", BOOTSTRAP)

ENVIRONMENT = check_environment(
    strict=True,
    output=ARTIFACTS,
    check_network=False,
    provenance=CONTEXT,
)

print(summarize_environment(ENVIRONMENT))
print(
    {
        "experiment_session_id": CONTEXT.experiment_session_id,
        "collection_id": CONTEXT.collection_id,
        "corpus_id": CONTEXT.corpus_id,
        "source_commit": CONTEXT.source_commit,
        "source_dirty": CONTEXT.source_dirty,
    }
)


In [ ]:
import random

PAYLOAD_MIB = (16, 32, 48, 64, 96, 128)
REPETITIONS = 5
MEASURED_SECONDS = 30.0
SAMPLING_INTERVAL_SECONDS = 1.0
TIMEOUT_SECONDS = 120.0
ORDER_SEED = 20260803

if len(PAYLOAD_MIB) != len(set(PAYLOAD_MIB)):
    raise RuntimeError("Payloads must be unique.")
if REPETITIONS < 5:
    raise RuntimeError("This study requires at least five repetitions.")
if MEASURED_SECONDS < 30:
    raise RuntimeError("Measured duration must be at least 30 seconds.")

RUN_SCHEDULE = []
for repetition in range(REPETITIONS):
    shuffled = list(PAYLOAD_MIB)
    random.Random(ORDER_SEED + repetition).shuffle(shuffled)
    RUN_SCHEDULE.append(
        {
            "repetition": repetition,
            "order": ["idle", *shuffled],
        }
    )

PLANNED_RUN_COUNT = REPETITIONS * (1 + len(PAYLOAD_MIB))

STUDY_PLAN = {
    "artifact_kind": "experiment_summary",
    "schema_version": SCHEMA_VERSION,
    "summary_type": "calibration_v4_sampling_study_plan",
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "payload_mib": list(PAYLOAD_MIB),
    "repetitions": REPETITIONS,
    "measured_seconds": MEASURED_SECONDS,
    "sampling_interval_seconds": SAMPLING_INTERVAL_SECONDS,
    "expected_minimum_sampling_opportunities_per_run": int(
        MEASURED_SECONDS / SAMPLING_INTERVAL_SECONDS
    ),
    "planned_run_count": PLANNED_RUN_COUNT,
    "order_seed": ORDER_SEED,
    "run_schedule": RUN_SCHEDULE,
    "threshold_policy": {
        "minimum_capture_rate": 0.8,
        "baseline_multiplier": 3.0,
        "baseline_floor_bytes_per_s": 1_000_000.0,
        "minimum_rank_correlation": 0.7,
        "minimum_dynamic_range": 1.2,
    },
    "not_a_standard_calibration_gate": True,
}

STORE.write_json("results/calibration-v4-study-plan.json", STUDY_PLAN)
print(STUDY_PLAN)


## Execute the sampling-resolution study

Run this cell **once**. It performs 35 dual-GPU runs and can take roughly 20–30 minutes.

Do not rerun this cell in the same workspace after an interruption. Preserve the partial artifacts and start a fresh notebook run instead.


In [ ]:
from commguard.orchestrator import run_experiment
from commguard.workloads import calibration_workload_name

RUN_MARKER = ARTIFACTS / "results" / "calibration-v4-execution-started.json"
if RUN_MARKER.exists():
    raise RuntimeError(
        "This study was already started in this workspace. "
        "Preserve current evidence and start a fresh notebook run."
    )

STORE.write_json(
    "results/calibration-v4-execution-started.json",
    {
        "artifact_kind": "experiment_summary",
        "schema_version": SCHEMA_VERSION,
        "summary_type": "calibration_v4_execution_started",
        "notebook_run_id": NOTEBOOK_RUN_ID,
        "planned_run_count": PLANNED_RUN_COUNT,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    },
)

OBSERVATIONS = []
RUN_INDEX = 0

def observation_from_outcome(
    outcome,
    *,
    observation_type,
    repetition,
    payload_mib,
    collective,
):
    manifest = outcome["manifest"]
    return {
        "run_id": outcome["run_id"],
        "run_directory": f"runs/{outcome['run_id']}",
        "workload_name": manifest["workload_name"],
        "worker_mode": manifest["config"]["mode"],
        "observation_type": observation_type,
        "is_idle": observation_type == "idle_baseline",
        "repetition": repetition,
        "payload_mib": payload_mib,
        "collective": collective,
        "participation_valid": manifest["participation_valid"],
        "exit_status": manifest["exit_status"],
        "pcie_supported": outcome["pcie_supported"],
        "pcie_total_mean_bytes_per_s": outcome[
            "pcie_total_mean_bytes_per_s"
        ],
        "pcie_total_median_bytes_per_s": outcome[
            "pcie_total_median_bytes_per_s"
        ],
        "pcie_sample_count": outcome["pcie_sample_count"],
        "measurement_phase_bounded": outcome[
            "measurement_phase_bounded"
        ],
        "measured_duration_seconds": manifest.get(
            "measured_duration_seconds"
        ),
    }

for schedule in RUN_SCHEDULE:
    repetition = schedule["repetition"]

    for scheduled_item in schedule["order"]:
        RUN_INDEX += 1
        print(
            {
                "run_index": RUN_INDEX,
                "planned_run_count": PLANNED_RUN_COUNT,
                "repetition": repetition,
                "scheduled_item": scheduled_item,
            },
            flush=True,
        )

        if scheduled_item == "idle":
            outcome = run_experiment(
                "calibration_idle",
                output=ARTIFACTS,
                overrides={
                    "repetition": repetition,
                    "min_measured_seconds": MEASURED_SECONDS,
                    "iteration_cap": None,
                    "idle_interval_s": 0.25,
                    "sampling_interval_s": SAMPLING_INTERVAL_SECONDS,
                },
                timeout_s=TIMEOUT_SECONDS,
                strict_preflight=True,
                raise_on_failure=False,
                provenance=CONTEXT,
            )
            observation = observation_from_outcome(
                outcome,
                observation_type="idle_baseline",
                repetition=repetition,
                payload_mib=0,
                collective=None,
            )
        else:
            payload = int(scheduled_item)
            workload_name = calibration_workload_name(
                "all_reduce", payload
            )
            outcome = run_experiment(
                workload_name,
                output=ARTIFACTS,
                overrides={
                    "repetition": repetition,
                    "payload_mib": payload,
                    "collective": "all_reduce",
                    "min_measured_seconds": MEASURED_SECONDS,
                    "iteration_cap": None,
                    "iterations": 1,
                    "burst_iterations": 10,
                    "iteration_interval_s": 0.25,
                    "sampling_interval_s": SAMPLING_INTERVAL_SECONDS,
                },
                timeout_s=TIMEOUT_SECONDS,
                strict_preflight=True,
                raise_on_failure=False,
                provenance=CONTEXT,
            )
            observation = observation_from_outcome(
                outcome,
                observation_type="collective",
                repetition=repetition,
                payload_mib=payload,
                collective="all_reduce",
            )

        OBSERVATIONS.append(observation)
        STORE.write_json(
            f"results/calibration-v4-observation-{RUN_INDEX:03d}.json",
            {
                "artifact_kind": "experiment_summary",
                "schema_version": SCHEMA_VERSION,
                "summary_type": "calibration_v4_observation",
                **observation,
            },
        )
        print(
            {
                "completed_run_index": RUN_INDEX,
                "run_id": observation["run_id"],
                "exit_status": observation["exit_status"],
                "pcie_sample_count": observation["pcie_sample_count"],
                "mean_bytes_per_s": observation[
                    "pcie_total_mean_bytes_per_s"
                ],
            },
            flush=True,
        )

print(
    {
        "execution_complete": True,
        "observations": len(OBSERVATIONS),
        "expected": PLANNED_RUN_COUNT,
    }
)


In [ ]:
from collections import Counter
from commguard.calibration import analyze_calibration

if len(OBSERVATIONS) != PLANNED_RUN_COUNT:
    raise RuntimeError(
        f"Incomplete study: {len(OBSERVATIONS)} of {PLANNED_RUN_COUNT} observations."
    )

run_ids = [row["run_id"] for row in OBSERVATIONS]
if len(run_ids) != len(set(run_ids)):
    raise RuntimeError("Duplicate run IDs detected.")

counts = Counter(
    (
        row["observation_type"],
        int(row["payload_mib"]),
    )
    for row in OBSERVATIONS
)

expected_counts = {
    ("idle_baseline", 0): REPETITIONS,
    **{
        ("collective", payload): REPETITIONS
        for payload in PAYLOAD_MIB
    },
}

if counts != Counter(expected_counts):
    raise RuntimeError(
        f"Observation matrix mismatch: actual={dict(counts)} "
        f"expected={expected_counts}"
    )

DIAGNOSTIC_ANALYSIS = analyze_calibration(
    OBSERVATIONS,
    minimum_sizes=len(PAYLOAD_MIB),
    minimum_rank_correlation=0.7,
    minimum_dynamic_range=1.2,
    minimum_repetitions=REPETITIONS,
    minimum_capture_rate=0.8,
    baseline_multiplier=3.0,
    baseline_floor_bytes_per_s=1_000_000.0,
)

failed_runs = [
    row for row in OBSERVATIONS
    if row["exit_status"] != "completed"
    or not row["participation_valid"]
]

short_runs = [
    row for row in OBSERVATIONS
    if (
        row.get("measured_duration_seconds") is None
        or row["measured_duration_seconds"] < MEASURED_SECONDS * 0.95
    )
]

low_sample_runs = [
    row for row in OBSERVATIONS
    if row["pcie_sample_count"] < 25
]

STUDY_VALIDATION = {
    "complete_observation_matrix": len(OBSERVATIONS) == PLANNED_RUN_COUNT,
    "unique_run_ids": len(run_ids) == len(set(run_ids)),
    "planned_run_count": PLANNED_RUN_COUNT,
    "observation_count": len(OBSERVATIONS),
    "idle_observation_count": counts[("idle_baseline", 0)],
    "payload_observation_counts": {
        str(payload): counts[("collective", payload)]
        for payload in PAYLOAD_MIB
    },
    "failed_run_count": len(failed_runs),
    "short_measured_run_count": len(short_runs),
    "low_sample_run_count": len(low_sample_runs),
    "all_runs_completed": not failed_runs,
    "all_runs_duration_valid": not short_runs,
    "all_runs_have_at_least_25_samples": not low_sample_runs,
    "diagnostic_only": True,
    "valid_as_standard_calibration_gate": False,
}

RESULT = {
    "artifact_kind": "experiment_summary",
    "schema_version": SCHEMA_VERSION,
    "summary_type": "calibration_v4_sampling_study",
    "notebook_version": NOTEBOOK_VERSION,
    "notebook_run_id": NOTEBOOK_RUN_ID,
    "source_commit": CONTEXT.source_commit,
    "experiment_session_id": CONTEXT.experiment_session_id,
    "collection_id": CONTEXT.collection_id,
    "corpus_id": CONTEXT.corpus_id,
    "environment_fingerprint": ENVIRONMENT["environment_fingerprint"],
    "study_plan": STUDY_PLAN,
    "observations": OBSERVATIONS,
    "analysis": DIAGNOSTIC_ANALYSIS,
    "study_validation": STUDY_VALIDATION,
    "interpretation_boundary": (
        "Diagnostic sampling-resolution evidence only. "
        "This result does not establish training detection and must not "
        "be used directly as the benign-corpus calibration gate."
    ),
}

RESULT_PATH = STORE.write_json(
    "results/calibration-v4-sampling-study.json",
    RESULT,
)

STORE.write_json(
    "results/calibration-v4-execution-completed.json",
    {
        "artifact_kind": "experiment_summary",
        "schema_version": SCHEMA_VERSION,
        "summary_type": "calibration_v4_execution_completed",
        "result_artifact": str(RESULT_PATH.relative_to(ARTIFACTS)),
        "study_validation": STUDY_VALIDATION,
        "diagnostic_status": DIAGNOSTIC_ANALYSIS.get("status"),
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    },
)

print(
    {
        "diagnostic_status": DIAGNOSTIC_ANALYSIS.get("status"),
        "decision_state": DIAGNOSTIC_ANALYSIS.get("decision_state"),
        "idle_usable_repetitions": DIAGNOSTIC_ANALYSIS.get(
            "idle_baseline_usable_repetitions"
        ),
        "capture_threshold_bytes_per_s": DIAGNOSTIC_ANALYSIS.get(
            "capture_threshold_bytes_per_s"
        ),
        "supported_payload_range_mib": DIAGNOSTIC_ANALYSIS.get(
            "supported_payload_range_mib"
        ),
        "unreliable_payload_range_mib": DIAGNOSTIC_ANALYSIS.get(
            "unreliable_payload_range_mib"
        ),
        "payload_summaries": DIAGNOSTIC_ANALYSIS.get(
            "payload_summaries"
        ),
        "study_validation": STUDY_VALIDATION,
        "result_path": str(RESULT_PATH),
    },
    flush=True,
)


## Interpretation gate

This notebook is a diagnostic study.

- If 16–48 MiB become reliably observable with 30-second sustained communication, the earlier v3 misses were likely dominated by sampling-duration or burst-alignment effects.
- If only 64 MiB and above remain reliable, document a large-message observability boundary for this hardware and telemetry method.
- If results remain non-monotonic, inspect NVML sampling timestamps, burst timing, and clock/power-state effects before changing thresholds.
- Do not lower the 0.8 capture-rate gate merely to obtain a pass.
- Do not start benign collection directly from this v4 archive. Review it first, then decide whether to update and rerun the canonical v3 gate or formally scope CommGuard to large sustained collectives.


In [ ]:
from commguard.artifacts import sha256_file

ARCHIVE = Path(
    f"/kaggle/working/"
    f"commguard-calibration-v4-sampling-study-{NOTEBOOK_RUN_ID}.tar.gz"
)
SHA_FILE = Path(f"{ARCHIVE}.sha256")

if ARCHIVE.exists() or SHA_FILE.exists():
    raise RuntimeError(
        "Refusing to overwrite an existing archive or checksum file."
    )

STORE.export(ARCHIVE)
ARCHIVE_SHA256 = sha256_file(ARCHIVE)

with SHA_FILE.open("x", encoding="utf-8") as stream:
    stream.write(f"{ARCHIVE_SHA256}  {ARCHIVE.name}\n")

print(
    {
        "archive": str(ARCHIVE),
        "archive_sha256": ARCHIVE_SHA256,
        "sha256_file": str(SHA_FILE),
        "result_artifact": str(RESULT_PATH.relative_to(ARTIFACTS)),
        "diagnostic_status": DIAGNOSTIC_ANALYSIS.get("status"),
        "valid_as_standard_calibration_gate": False,
    },
    flush=True,
)

print("\nDOWNLOAD THESE THREE FILES:")
print(f"1. {ARCHIVE}")
print(f"2. {SHA_FILE}")
print("3. This executed notebook")
print(
    "\nDo not run the benign-corpus notebook until this evidence "
    "has been reviewed and a standard calibration decision has been made."
)
